# 08 流水线并行

## 简介

本笔记本介绍流水线并行（Pipeline Parallelism, PP）的两种经典调度策略：
GPipe 和 1F1B（One Forward One Backward）。
流水线并行将 Transformer 层按深度切分到多张 GPU，每张 GPU 负责一部分连续层的计算，
通过 micro-batch 调度来减小流水线气泡（bubble time）。

In [ ]:
import torch
import sys; sys.path.insert(0, '..')
import matplotlib.pyplot as plt
from parallel.pipeline_parallel.layer_partition import get_layer_range, partition_layers
from parallel.pipeline_parallel.gpiped import gpiped_forward, compute_gpipe_bubble_time
from parallel.pipeline_parallel.f1b1 import f1b1_schedule, compute_1f1b_bubble_time

print("流水线并行模块加载完成")

## 1. 层切分（Layer Partitioning）

流水线并行的第一步是将模型的 Transformer 层按深度均匀分配到各 GPU。

```
  GPU 0        GPU 1        GPU 2        GPU 3
  ──────       ──────       ──────       ──────
  Layer 0      Layer 3      Layer 6      Layer 9
  Layer 1      Layer 4      Layer 7      Layer 10
  Layer 2      Layer 5      Layer 8      Layer 11

  (12 层 Transformer 分配到 4 张 GPU, 每张 3 层)
```

**关键问题**: 如果层数不能整除 GPU 数，如何处理？
常见做法是前面的 GPU 多分配 1 层来吸收余数（因为前面的 GPU 在流水线中空闲更少）。

In [ ]:
# 层切分演示
def demo_layer_partition(n_layers, world_size):
    print(f"\n=== {n_layers} 层分配到 {world_size} 张 GPU ===")
    layers_per_rank = []
    for rank in range(world_size):
        start, end = get_layer_range(n_layers, rank, world_size)
        n = end - start
        layers_per_rank.append(n)
        print(f"  Rank {rank}: Layer [{start}:{end}) = {n} 层")
    print(f"  均匀度: min={min(layers_per_rank)}, max={max(layers_per_rank)}, diff={max(layers_per_rank)-min(layers_per_rank)}")

demo_layer_partition(12, 4)    # 均匀
demo_layer_partition(32, 8)    # 均匀
demo_layer_partition(13, 4)    # 非均匀：余 1 层
demo_layer_partition(40, 8)    # 均匀 (40/8=5)
demo_layer_partition(28, 6)    # 非均匀：余 4 层

# 常见模型层数参考
print("\n" + "=" * 50)
print("常见模型层数与 PP 配置建议:")
print("=" * 50)
models = [
    ("Llama-7B", 32, 4),
    ("Llama-13B", 40, 4),
    ("Llama-70B", 80, 8),
    ("DeepSeek-V3", 61, 8),
]
for name, n_layers, pp_size in models:
    layers_per = n_layers // pp_size
    remainder = n_layers % pp_size
    print(f"  {name:15s}: {n_layers:2d} 层 / {pp_size} GPU = {layers_per} 层/卡", end="")
    if remainder:
        print(f" (余 {remainder} 层)")
    else:
        print(" (均匀)")

### 🧠 直觉理解：流水线并行

流水线并行就像**工厂流水线**：一条产品线分成 4 道工序（4 个 GPU），每道工序处理半成品然后传给下一道。问题是：第一道工序做完第一批后，要等最后一道工序做完才能开始反向传播——中间的等待时间就是"气泡"。

**GPipe vs 1F1B**：GPipe 像是先把所有产品都做到一半（全部 forward），再从最后一道开始往回返工（全部 backward）——中间积压大量半成品（激活显存高）。1F1B 则是做完一个产品的 forward 立刻做 backward——半成品积压少，但调度更复杂。

### 📐 数学原理：气泡时间公式

**GPipe 气泡比例**：

$$\text{Bubble}_{\text{GPipe}} = \frac{P - 1}{P - 1 + M}$$

**1F1B 气泡比例**：

$$\text{Bubble}_{\text{1F1B}} = \frac{2(P-1)}{2(P - 1 + M) - 1}$$

其中 $P$ = stage 数（GPU 数），$M$ = micro-batch 数。

**1F1B 激活显存峰值**：仅需存储 warmup 阶段 $P-1$ 个 micro-batch 的激活，
远小于 GPipe 的 $M$ 个 micro-batch。

当 $M \gg P$ 时，两种策略的气泡都趋近于 0。

## 2. GPipe 调度策略

GPipe 是最简单的流水线策略：
1. **Warmup**: 所有 micro-batch 依次做 forward，填充流水线
2. **Cooldown**: 所有 micro-batch 依次做 backward，排空流水线

```
时间 →
GPU 0:  F0  F1  F2  F3                   B0  B1  B2  B3
GPU 1:      F0  F1  F2  F3           B0  B1  B2  B3
GPU 2:          F0  F1  F2  F3   B0  B1  B2  B3
GPU 3:              F0  F1  F2  F3  B0  B1  B2  B3
                    <--- Bubble --->
```

**气泡时间公式**:
$$Bubble\_ratio = \frac{P - 1}{P - 1 + M}$$

其中 $P$ = stage 数 (GPU 数), $M$ = micro-batch 数。

- $M$ 越大，气泡比例越小
- $P$ 越大，气泡比例越大
- 代价: $M$ 大 → 激活显存峰值高（需存储所有 micro-batch 的中间激活）

In [ ]:
# GPipe 气泡时间分析
print("GPipe Bubble Time 分析")
print("公式: bubble = (P-1) / (P-1 + M)")
print()

# 表格展示不同配置的气泡比例
stages_list = [2, 4, 8, 16]
micro_batches_list = [4, 8, 16, 32, 64]

print(f"{'M\\P':>6s}", end="")
for P in stages_list:
    print(f"{f'P={P}':>10s}", end="")
print()
print("-" * (6 + 10 * len(stages_list)))

for M in micro_batches_list:
    print(f"{M:>6d}", end="")
    for P in stages_list:
        bubble = compute_gpipe_bubble_time(M, P)
        print(f"{bubble:>9.1%}", end=" ")
    print()

print(f"\n★ M={micro_batches_list[-1]} 时，即使 P=16，气泡也仅 {compute_gpipe_bubble_time(micro_batches_list[-1], 16):.1%}")
print(f"★ 但激活显存随 M 线性增长 → GPipe 不适合大 M")

### 📊 GPipe/1F1B 时间线甘特图

用 matplotlib barh 绘制 GPipe 和 1F1B 的时间线甘特图，直观展示气泡和显存差异。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

P = 4  # stages
M = 4  # micro-batches

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# GPipe 甘特图
ax = axes[0]
for stage in range(P):
    y = P - 1 - stage
    # Forward passes
    for mb in range(M):
        start = mb + stage
        ax.barh(y, 0.9, left=start + 0.05, color='#3498db', edgecolor='black', linewidth=0.5)
        ax.text(start + 0.5, y, f'F{mb}', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    # Backward passes
    for mb in range(M):
        start = M + (P - 1 - stage) + mb
        ax.barh(y, 0.9, left=start + 0.05, color='#e74c3c', edgecolor='black', linewidth=0.5)
        ax.text(start + 0.5, y, f'B{mb}', ha='center', va='center', fontsize=8, color='white', fontweight='bold')

# 标记气泡
for stage in range(P):
    y = P - 1 - stage
    bubble_start = M + stage
    bubble_end = M + (P - 1 - stage)
    if bubble_end > bubble_start:
        ax.barh(y, bubble_end - bubble_start, left=bubble_start + 0.05,
                color='#bdc3c7', alpha=0.3, edgecolor='gray', linestyle='--')

ax.set_yticks(range(P))
ax.set_yticklabels([f'GPU {P-1-i}' for i in range(P)])
ax.set_xlabel('时间步', fontsize=12)
ax.set_title('GPipe 调度时间线', fontsize=14, fontweight='bold')
ax.set_xlim(-0.5, 2*M + 2*P)

# 1F1B 甘特图
ax2 = axes[1]
for stage in range(P):
    y = P - 1 - stage
    t = stage  # 当前时间步
    # Warmup: P-1-stage 个 forward
    warmup_count = P - 1 - stage
    for mb in range(warmup_count + 1):
        ax2.barh(y, 0.9, left=t + 0.05, color='#3498db', edgecolor='black', linewidth=0.5)
        ax2.text(t + 0.5, y, f'F{mb}', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
        t += 1
    # Steady: 1F1B
    for mb in range(warmup_count + 1, M):
        ax2.barh(y, 0.9, left=t + 0.05, color='#3498db', edgecolor='black', linewidth=0.5)
        ax2.text(t + 0.5, y, f'F{mb}', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
        t += 1
        ax2.barh(y, 0.9, left=t + 0.05, color='#e74c3c', edgecolor='black', linewidth=0.5)
        ax2.text(t + 0.5, y, f'B{mb - warmup_count}', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
        t += 1
    # Cooldown
    for mb in range(M - warmup_count, M):
        ax2.barh(y, 0.9, left=t + 0.05, color='#e74c3c', edgecolor='black', linewidth=0.5)
        ax2.text(t + 0.5, y, f'B{mb}', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
        t += 1

ax2.set_yticks(range(P))
ax2.set_yticklabels([f'GPU {P-1-i}' for i in range(P)])
ax2.set_xlabel('时间步', fontsize=12)
ax2.set_title('1F1B 调度时间线', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. 1F1B 调度策略

1F1B (One Forward One Backward) 通过交错 forward 和 backward 来减少激活显存峰值：

1. **Warmup**: 先连续做 $P-1$ 个 forward，填充流水线
2. **Steady**: 每个 step 交替做 1 个 forward + 1 个 backward
3. **Cooldown**: 处理剩余的 backward

```
时间 →
GPU 0:  F0  F1  F2  F3  B0  F4  B1  F5  B2  F6  B3  B4  B5  B6
GPU 1:      F0  F1  F2  F3  B0  F4  B1  F5  B2  F6  B3  B4  B5  B6
                       <-- Steady State: 1F + 1B alternating -->
```

**气泡时间公式**:
$$Bubble\_ratio = \frac{2(P-1)}{2(P-1+M)-1}$$

- 比 GPipe 气泡稍大（因为有 warmup 和 cooldown 两端的空闲）
- 但激活显存峰值远小于 GPipe（只有 warmup 阶段的 $P-1$ 个 micro-batch 的激活同时在存）

In [ ]:
# 1F1B vs GPipe 气泡时间对比
print("1F1B vs GPipe 气泡时间对比")
print("=" * 70)

stages_list = [4, 8]
micro_batches_list = [4, 8, 16, 32, 64]

for P in stages_list:
    print(f"\n--- P = {P} stages ---")
    print(f"{'M':>6s}  {'GPipe Bubble':>14s}  {'1F1B Bubble':>14s}  {'Diff':>8s}  {'1F1B显存优势'}")
    print("-" * 65)
    for M in micro_batches_list:
        b_gpipe = compute_gpipe_bubble_time(M, P)
        b_1f1b = compute_1f1b_bubble_time(M, P)
        diff = b_1f1b - b_gpipe
        # 激活显存优势: 1F1B 最多存 P-1 个 micro-batch 的激活, GPipe 存 M 个
        mem_ratio = (P - 1) / M
        print(f"{M:>6d}  {b_gpipe:>13.1%}  {b_1f1b:>13.1%}  {diff:>+7.1%}  {mem_ratio:.0%}")

print("\n★ 1F1B 的气泡比 GPipe 略大，但显存优势显著 (尤其大 M 时)")
print("★ 大模型训练几乎都用 1F1B (或变种如 interleaved 1F1B)")

In [ ]:
# 可视化: 气泡时间随 M 的变化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图: 气泡 vs M (不同 P)
M_values = list(range(2, 65))
for P in [2, 4, 8, 16]:
    gpipe_bubbles = [compute_gpipe_bubble_time(M, P) for M in M_values]
    axes[0].plot(M_values, gpipe_bubbles, label=f'GPipe P={P}', linewidth=2)
    f1b1_bubbles = [compute_1f1b_bubble_time(M, P) for M in M_values]
    axes[0].plot(M_values, f1b1_bubbles, '--', label=f'1F1B P={P}', linewidth=2, alpha=0.7)

axes[0].set_xlabel('Micro-batch 数量 M', fontsize=12)
axes[0].set_ylabel('气泡比例', fontsize=12)
axes[0].set_title('气泡时间 vs Micro-batch 数量', fontsize=14)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 0.6)

# 右图: 气泡 vs P (不同 M)
P_values = list(range(2, 33))
for M in [4, 16, 64]:
    gpipe_bubbles = [compute_gpipe_bubble_time(M, P) for P in P_values]
    axes[1].plot(P_values, gpipe_bubbles, label=f'GPipe M={M}', linewidth=2)
    f1b1_bubbles = [compute_1f1b_bubble_time(M, P) for P in P_values]
    axes[1].plot(P_values, f1b1_bubbles, '--', label=f'1F1B M={M}', linewidth=2, alpha=0.7)

axes[1].set_xlabel('Stage 数 P (GPU 数)', fontsize=12)
axes[1].set_ylabel('气泡比例', fontsize=12)
axes[1].set_title('气泡时间 vs Pipeline Stage 数', fontsize=14)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("关键观察:")
print("1. M 越大 → 气泡越小 (GPipe 和 1F1B 都趋近于 0)")
print("2. P 越大 → 气泡越大 (需要更多 M 来填充)")
print("3. 1F1B 气泡始终略大于 GPipe，但显存优势远超气泡差距")
print("4. 实践中 M=32~64 是常见的平衡点")

### 📊 Bubble 对比图

对比不同 micro-batch 数量下 GPipe 和 1F1B 的气泡比例。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图: 不同 M 下的气泡比例
M_values = np.arange(2, 65)
for P in [2, 4, 8, 16]:
    gpipe = [(P-1)/(P-1+M) for M in M_values]
    f1b1 = [2*(P-1)/(2*(P-1+M)-1) for M in M_values]
    axes[0].plot(M_values, gpipe, '-', label=f'GPipe P={P}', linewidth=2)
    axes[0].plot(M_values, f1b1, '--', label=f'1F1B P={P}', linewidth=2, alpha=0.7)

axes[0].set_xlabel('Micro-batch 数量 M', fontsize=12)
axes[0].set_ylabel('气泡比例', fontsize=12)
axes[0].set_title('气泡比例 vs M', fontsize=14)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 0.6)

# 右图: 不同 M 下的激活显存峰值比
M_peak = np.arange(2, 65)
for P in [4, 8, 16]:
    # GPipe: 存 M 个 micro-batch 的激活
    # 1F1B: 存 P-1 个 micro-batch 的激活
    mem_ratio = [(P-1)/M for M in M_peak]
    axes[1].plot(M_peak, mem_ratio, label=f'1F1B/GPipe P={P}', linewidth=2)

axes[1].axhline(y=1.0, color='red', linestyle=':', linewidth=1, label='1F1B=GPipe')
axes[1].set_xlabel('Micro-batch 数量 M', fontsize=12)
axes[1].set_ylabel('1F1B/GPipe 激活显存比', fontsize=12)
axes[1].set_title('1F1B 相对 GPipe 的激活显存优势', fontsize=14)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1.2)

plt.tight_layout()
plt.show()

## 4. 关键要点总结

1. **层切分**: 将 Transformer 层按深度分配，前面 GPU 多分配余数层
2. **GPipe**: 所有 forward 完成后再 backward，简单但激活显存峰值高（需存 M 个 micro-batch 的中间激活）
3. **1F1B**: Forward 和 Backward 交替进行，激活显存仅 peak 在 warmup 阶段（P-1 个 micro-batch）
4. **气泡公式**: GPipe bubble = (P-1)/(P-1+M), 1F1B bubble ≈ 2(P-1)/[2(P-1+M)-1]
5. **Micro-batch 数量**: M 越大气泡越小，但显存和调度开销增加
6. **PP 的局限**: 不可避免的气泡时间（bubble time）是 PP 的固有代价
7. **组合使用**: PP 通常与 TP（节点内）+ DP（跨节点）组合，形成 3D 并行策略

## 📝 练习题

### 思考题
1. 为什么 1F1B 的气泡比例比 GPipe 略大？多出的气泡来自哪里？

### 编程题
2. 实现一个 1F1B 调度模拟器：给定 P=4, M=8，打印每个时间步每个 GPU 正在执行的操作（F0, B1, idle 等），并计算实际气泡比例。

### 分析题
3. 一个 70B 模型有 80 层，使用 PP=8（每卡 10 层），每层 forward 耗时 10ms，backward 耗时 20ms。计算 M=16 时 GPipe 和 1F1B 的总训练时间和气泡比例。如果 micro-batch 的激活显存为 200MB/个，两种策略的峰值激活显存分别是多少？